# NB07 — Model tests: is my generator correct, and is it any good?

**Project 1.1 — checks to run around NB03 / NB04 / NB05.**

Simple idea: **a model that "looks fine" can still be wrong.** So we test in layers, cheapest first.

| Part | Question | Needs a trained model? |
|---|---|---|
| 1 | Is the code correct? (mask, loss, sampling, generation) | No |
| 2 | Can the model learn at all? (overfit one batch) | No |
| 3 | Does the model beat dumb baselines? | Yes (checkpoints) |
| 4 | Are the generated sequences good and *new*? | Yes |
| 5 | Do they look like real VHH? | Yes |

Every test calls `check(...)`. At the end you get one PASS / FAIL table.
**If a Part 1 or Part 2 test fails, stop. Do not train or generate until it is fixed.**

Input: `train.csv` and `val.csv` from NB02 (same `DATA_VERSION`). Checkpoints (optional): `models/lstm_best.pt`, `models/transformer_best.pt`.

In [ ]:
# ==============================================================================
# Cell 1 : Bootstrap (local AND Google Colab), imports, settings
# ==============================================================================
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AhmadBelhaj4/Recherche_AI.git"
BRANCH = "main"


def _find_root(start):
    """Walk up from `start` until we find the folder that contains utils.py."""
    start = Path(start).resolve()
    for p in [start, *start.parents]:
        if (p / "utils.py").exists():
            return p
        if (p / "project1" / "utils.py").exists():
            return p / "project1"
    return None


ROOT_DIR = _find_root(Path.cwd())

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if ROOT_DIR is None and IN_COLAB:          # Colab
    clone_target = Path("/content/Recherche_AI")
    if not clone_target.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(clone_target)], check=True)
    ROOT_DIR = _find_root(clone_target)

if ROOT_DIR is None:
    raise FileNotFoundError("utils.py not found: open this notebook inside the Recherche_AI repo.")

if IN_COLAB:                                                # Colab only: mount Drive BEFORE get_paths()
    from google.colab import drive
    drive.mount("/content/drive")
    _drive = Path("/content/drive/MyDrive/AI_Project_Shared")
    if not _drive.exists():
        raise FileNotFoundError(f"Drive folder not found: {_drive}. Add the shortcut to My Drive.")
    os.environ.setdefault("AI_PROJECT_DIR", str(_drive))

sys.path.insert(0, str(ROOT_DIR))

import math, random, re, copy
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from utils import (get_paths, get_data_version, set_seed, get_device, AminoAcidTokenizer, make_loader, shift,
                   is_valid_sequence, compute_nats_and_perplexity, compute_aa_composition,
                   STANDARD_AA, PAD_ID, BOS_ID, EOS_ID, VOCAB_SIZE, MIN_LEN, MAX_LEN)

# ---- settings (change here only) ----
SEED = 42
VAL_MAX = 20_000            # max validation sequences used for scoring (keeps it fast)
BASELINE_TRAIN_MAX = 200_000  # max train sequences used to count the unigram/bigram baselines
OVERFIT_STEPS = 400         # Part 2
N_GEN = 500                 # sequences generated per sampling config (use 10_000 only for the final run)
NN_TRAIN_SAMPLE = 20_000    # train sequences searched for the "nearest copy" test (Part 4)
MODEL_TO_GENERATE = "transformer"   # "transformer" or "lstm"
ANARCI_N = 100             # sequences per set sent to ANARCI (optional, Part 5)

set_seed(SEED)
device = get_device()
tokenizer = AminoAcidTokenizer()

PATHS = get_paths()
DATA_VERSION = get_data_version()
DATA_DIR = PATHS["processed"] / DATA_VERSION
OUT_DIR = PATHS["results"] / "model_tests" / DATA_VERSION
OUT_DIR.mkdir(parents=True, exist_ok=True)

for name in ("train.csv", "val.csv"):
    if not (DATA_DIR / name).exists():
        raise FileNotFoundError(f"{DATA_DIR / name} not found: run NB01 then NB02 first (same DATA_VERSION).")

read = lambda n: pd.read_csv(DATA_DIR / n, dtype=str, keep_default_na=False)["sequence"].tolist()
train_seqs, val_seqs_all = read("train.csv"), read("val.csv")
rng_np = np.random.default_rng(SEED)
val_idx = rng_np.permutation(len(val_seqs_all))[:VAL_MAX]
val_seqs = [val_seqs_all[i] for i in val_idx]

# ---- the PASS / FAIL register ----
CHECKS = []


def check(name, ok, detail="", kind="must"):
    """kind='must'   -> FAIL if not ok (a bug)
       kind='should' -> WARN if not ok (suspicious, look at it)"""
    status = "PASS" if ok else ("FAIL" if kind == "must" else "WARN")
    CHECKS.append({"check": name, "status": status, "detail": detail})
    print(f"[{status}] {name}" + (f"  ({detail})" if detail else ""))
    return bool(ok)


print(f"MODE: {'REAL DATA' if DATA_VERSION != 'toy' else 'TOY (fake data)'} | device: {device} | seed: {SEED}")
print(f"train: {len(train_seqs):,} | val used: {len(val_seqs):,} (of {len(val_seqs_all):,})")


In [ ]:
# ==============================================================================
# Cell 2 : Model builders (use models.py if it works, else an inline copy of NB03/NB04)
# ==============================================================================
class LSTMLM_inline(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.proj = nn.Linear(hidden_dim, embed_dim) if hidden_dim != embed_dim else nn.Identity()
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.head.weight = self.embed.weight                      # weight tying

    def forward(self, x, hidden=None):
        out, hidden = self.lstm(self.embed(x), hidden)
        return self.head(self.proj(out)), hidden


class CausalTransformerLM_inline(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=4, num_layers=6, dim_feedforward=512,
                 max_len=200, dropout=0.1):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=dim_feedforward,
                                           dropout=dropout, activation="gelu", batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.head.weight = self.token_embed.weight

    def forward(self, x):
        b, t = x.shape
        pos = torch.arange(t, device=x.device).unsqueeze(0).expand(b, -1)
        mask = nn.Transformer.generate_square_subsequent_mask(t, device=x.device)
        out = self.transformer(self.token_embed(x) + self.pos_embed(pos), mask=mask, is_causal=True)
        return self.head(self.norm(out))


def build_lstm():
    try:
        from models import LSTMLM
        return LSTMLM(VOCAB_SIZE, 128, 256, 1).to(device), "models.py"
    except Exception:
        return LSTMLM_inline(VOCAB_SIZE, 128, 256, 1).to(device), "inline copy"


def build_transformer():
    try:
        from models import CausalTransformerLM
        return CausalTransformerLM(VOCAB_SIZE, 128, 4, 6, 512).to(device), "models.py"
    except Exception:
        return CausalTransformerLM_inline(VOCAB_SIZE, 128, 4, 6, 512).to(device), "inline copy"


def logits_of(model, x):
    """LSTM returns (logits, hidden), transformer returns logits. Always give back logits."""
    out = model(x)
    return out[0] if isinstance(out, tuple) else out


def disable_dropout(model):
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.p = 0.0
        if isinstance(m, nn.MultiheadAttention):
            m.dropout = 0.0


@torch.no_grad()
def mean_nats(model, seqs, batch_size=128):
    """EXACT nats/token = sum of NLL / number of real (non-PAD) tokens. Not a mean of batch means."""
    model.eval()
    total, n = 0.0, 0
    for batch in make_loader(seqs, tokenizer, batch_size=batch_size, shuffle=False):
        inp, tgt = shift(batch.to(device))
        logits = logits_of(model, inp)
        total += F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1),
                                 ignore_index=PAD_ID, reduction="sum").item()
        n += (tgt != PAD_ID).sum().item()
    return compute_nats_and_perplexity(total, n)


_, src_l = build_lstm()
_, src_t = build_transformer()
print(f"LSTM class comes from: {src_l} | Transformer class comes from: {src_t}")

## Part 1 — Is the code correct? (no training needed)

Each test has an answer we know in advance. If the code gives another answer, the code has a bug.

In [ ]:
# ==============================================================================
# Cell 3 : Tokenizer, causal mask, loss ignores PAD, initial loss
# ==============================================================================
# --- 3a. tokenizer round trip on 1000 random sequences
rnd = random.Random(SEED)
rand_seqs = ["".join(rnd.choices(STANDARD_AA, k=rnd.randint(MIN_LEN, MAX_LEN))) for _ in range(1000)]
check("tokenizer: round trip on 1000 random sequences",
      all(tokenizer.decode(tokenizer.encode(s)) == s for s in rand_seqs))

# --- 3b. causal mask: changing FUTURE tokens must not change PAST logits
for name, builder in (("transformer", build_transformer), ("lstm", build_lstm)):
    m, _ = builder()
    m.eval()
    x1 = torch.randint(0, 20, (4, 30), device=device)
    x2 = x1.clone()
    x2[:, 20:] = (x1[:, 20:] + 1) % 20                      # every future token is different
    with torch.no_grad():
        l1, l2 = logits_of(m, x1), logits_of(m, x2)
    past = (l1[:, :20] - l2[:, :20]).abs().max().item()
    future = (l1[:, 20:] - l2[:, 20:]).abs().max().item()
    check(f"{name}: past logits do not depend on future tokens", past < 1e-5, f"max diff {past:.1e}")
    check(f"{name}: test can detect a change (future logits DO change)", future > 1e-6, f"max diff {future:.1e}")

# --- 3c. the loss ignores PAD: adding 10 PAD columns must not change the summed loss
batch = next(iter(make_loader(train_seqs[:16], tokenizer, batch_size=16, shuffle=False))).to(device)
inp, tgt = shift(batch)
extra = torch.full((inp.size(0), 10), PAD_ID, dtype=torch.long, device=device)
inp_p, tgt_p = torch.cat([inp, extra], 1), torch.cat([tgt, extra], 1)
for name, builder in (("transformer", build_transformer), ("lstm", build_lstm)):
    m, _ = builder()
    m.eval()
    with torch.no_grad():
        a = F.cross_entropy(logits_of(m, inp).reshape(-1, VOCAB_SIZE), tgt.reshape(-1), ignore_index=PAD_ID, reduction="sum")
        b = F.cross_entropy(logits_of(m, inp_p).reshape(-1, VOCAB_SIZE), tgt_p.reshape(-1), ignore_index=PAD_ID, reduction="sum")
    check(f"{name}: extra PAD tokens do not change the loss", abs(a.item() - b.item()) < 1e-2,
          f"{a.item():.3f} vs {b.item():.3f}")

# --- 3d. loss of an UNTRAINED model
# With tied embeddings and PyTorch's default init (std = 1) the first loss is far above ln(23).
# With a small init (std = 0.02, as in GPT-2) it must be close to ln(23) = log of "all 23 tokens equally likely".
ln23 = math.log(VOCAB_SIZE)
print(f"\nReference: ln(23) = {ln23:.3f}")
for name, builder in (("transformer", build_transformer), ("lstm", build_lstm)):
    m, _ = builder()
    default_loss, _ = mean_nats(m, train_seqs[:256])
    for mod in m.modules():
        if isinstance(mod, nn.Embedding):
            nn.init.normal_(mod.weight, std=0.02)
    small_loss, _ = mean_nats(m, train_seqs[:256])
    print(f"  {name}: default init -> {default_loss:.3f} nats | small init (std 0.02) -> {small_loss:.3f} nats")
    check(f"{name}: untrained loss with small init is close to ln(23)", abs(small_loss - ln23) < 0.3, f"{small_loss:.3f}")
    check(f"{name}: default init start loss is reasonable (< 3 x ln 23)", default_loss < 3 * ln23,
          f"{default_loss:.2f} - if this warns, re-init embeddings with std 0.02 before training", kind="should")

In [ ]:
# ==============================================================================
# Cell 4 : Sampling functions (exact formulas) + tests on hand-made distributions
#          >>> When all tests pass, copy filter_probs / sample_next / generate into NB05 <<<
# ==============================================================================
def filter_probs(logits, temperature=1.0, top_k=0, top_p=0.0, blocked_ids=()):
    """logits (B, V) -> EXACT renormalised probabilities (B, V).
       1) blocked tokens get probability 0     2) p = softmax(z / T)
       3) top-k: keep the k best, renormalise  4) nucleus: keep token i if (cum_i - p_i) < top_p, renormalise"""
    assert temperature > 0, "temperature must be > 0 (use greedy=True for argmax)"
    z = logits.clone().float()
    for i in blocked_ids:
        z[:, i] = float("-inf")
    probs = F.softmax(z / temperature, dim=-1)

    if top_k > 0:
        k = min(top_k, probs.size(-1))
        idx = torch.topk(probs, k, dim=-1).indices
        keep = torch.zeros_like(probs, dtype=torch.bool).scatter_(1, idx, True)
        probs = torch.where(keep, probs, torch.zeros_like(probs))
        probs = probs / probs.sum(-1, keepdim=True)

    if 0.0 < top_p < 1.0:
        sp, si = torch.sort(probs, dim=-1, descending=True)
        cum = torch.cumsum(sp, dim=-1)
        keep_sorted = (cum - sp) < top_p                       # exact definition, no epsilon
        keep = torch.zeros_like(keep_sorted).scatter_(1, si, keep_sorted)
        probs = torch.where(keep, probs, torch.zeros_like(probs))
        probs = probs / probs.sum(-1, keepdim=True)
    return probs


def sample_next(logits, greedy=False, temperature=1.0, top_k=0, top_p=0.0, blocked_ids=(), generator=None):
    if greedy:
        z = logits.clone().float()
        for i in blocked_ids:
            z[:, i] = float("-inf")
        return z.argmax(dim=-1, keepdim=True)
    probs = filter_probs(logits, temperature, top_k, top_p, blocked_ids)
    return torch.multinomial(probs, 1, generator=generator)


# ---------------- tests ----------------
lg = torch.log(torch.tensor([[0.5, 0.3, 0.1, 0.1]]))
close = lambda a, b: torch.allclose(a, torch.tensor([b]), atol=1e-6)

check("nucleus p=0.7 keeps {0.5, 0.3} and renormalises exactly",
      close(filter_probs(lg, top_p=0.7), [0.625, 0.375, 0.0, 0.0]))
check("nucleus p=0.45 keeps only the top token (0.5 already reaches p)",
      close(filter_probs(lg, top_p=0.45), [1.0, 0.0, 0.0, 0.0]))
check("top-k=2 keeps 2 tokens and renormalises exactly",
      close(filter_probs(lg, top_k=2), [0.625, 0.375, 0.0, 0.0]))
check("temperature T=2 equals softmax(z/2)",
      torch.allclose(filter_probs(lg, temperature=2.0), F.softmax(lg / 2.0, dim=-1), atol=1e-6))
check("greedy = argmax", sample_next(lg, greedy=True).item() == 0)
bl = filter_probs(torch.zeros(1, 5), blocked_ids=(3, 4))
check("blocked tokens get probability 0 and the rest sums to 1",
      bl[0, 3].item() == 0 and bl[0, 4].item() == 0 and abs(bl.sum().item() - 1) < 1e-6)

z = torch.randn(6, VOCAB_SIZE) * 2
pk = filter_probs(z, top_k=10)
check("top-k=10 on random logits keeps exactly 10 tokens per row", bool(((pk > 0).sum(1) == 10).all()))

P = 0.9
base = F.softmax(z, dim=-1)
pn = filter_probs(z, top_p=P)
kept = pn > 0
mass = (base * kept).sum(1)
smallest = torch.where(kept, base, torch.full_like(base, 2.0)).min(1).values
check("nucleus: kept mass >= p, and it is the SMALLEST such set",
      bool((mass >= P - 1e-6).all()) and bool(((mass - smallest) < P + 1e-6).all()))

g = torch.Generator().manual_seed(0)
draws = sample_next(lg.expand(20000, -1), generator=g).squeeze(1)
freq = torch.bincount(draws, minlength=4).float() / 20000
check("sampling frequencies match [0.5, 0.3, 0.1, 0.1] (20 000 draws)",
      torch.allclose(freq, torch.tensor([0.5, 0.3, 0.1, 0.1]), atol=0.02), str([round(v, 3) for v in freq.tolist()]))

In [ ]:
# ==============================================================================
# Cell 5 : Generation function + tests (stops at EOS, PAD after EOS, seeds, no special tokens)
# ==============================================================================
@torch.no_grad()
def generate(model, n, max_new=MAX_LEN + 1, batch_size=64, seed=0, **cfg):
    """Start from BOS, sample until EOS (or max_new steps). Fixed, distinct seed per batch.
       Returns dict: seqs (strings), reached_eos (bool list), raw (token ids after BOS)."""
    model.eval()
    seqs, reached, raw = [], [], []
    for bi, start in enumerate(range(0, n, batch_size)):
        b = min(batch_size, n - start)
        g = torch.Generator(device=device)
        g.manual_seed(seed + bi)
        tokens = torch.full((b, 1), BOS_ID, dtype=torch.long, device=device)
        finished = torch.zeros(b, dtype=torch.bool, device=device)
        for _ in range(max_new):
            logits = logits_of(model, tokens)[:, -1, :]
            nxt = sample_next(logits, blocked_ids=(PAD_ID, BOS_ID), generator=g, **cfg)
            nxt = torch.where(finished.unsqueeze(1), torch.full_like(nxt, PAD_ID), nxt)   # no change after EOS
            tokens = torch.cat([tokens, nxt], dim=1)
            finished = finished | (nxt.squeeze(1) == EOS_ID)
            if finished.all():
                break
        for row in tokens.cpu().tolist():
            raw.append(row[1:])
            reached.append(EOS_ID in row)
            seqs.append(tokenizer.decode(row))
    return {"seqs": seqs, "reached_eos": reached, "raw": raw}


class EosAfterFive(nn.Module):
    """Fake model for testing: uniform logits, but after BOS + 5 residues it wants EOS."""
    def forward(self, x):
        out = torch.zeros(x.size(0), x.size(1), VOCAB_SIZE, device=x.device)
        out[:, :5, EOS_ID] = -20.0      # no EOS before 5 residues
        out[:, 5:, EOS_ID] = 20.0
        return out


fake = EosAfterFive().to(device)
for label, cfg in (("greedy", dict(greedy=True)), ("top-k", dict(top_k=3)), ("nucleus", dict(top_p=0.9))):
    r = generate(fake, 20, max_new=30, **cfg)
    check(f"generate ({label}): stops exactly at EOS (all length 5)", all(len(s) == 5 for s in r["seqs"]))
    after = all(all(t == PAD_ID for t in row[row.index(EOS_ID) + 1:]) for row in r["raw"])
    check(f"generate ({label}): only PAD after EOS", after)
    check(f"generate ({label}): never emits PAD or BOS before EOS",
          all(t not in (PAD_ID, BOS_ID) for row in r["raw"] for t in row[:row.index(EOS_ID)]))

tm, _ = build_transformer()
a = generate(tm, 8, max_new=25, seed=7, top_p=0.95)["seqs"]
b = generate(tm, 8, max_new=25, seed=7, top_p=0.95)["seqs"]
c = generate(tm, 8, max_new=25, seed=8, top_p=0.95)["seqs"]
check("same seed -> identical sequences", a == b)
check("different seed -> different sequences", a != c, kind="should")

## Part 2 — Can the model learn at all? (overfit ONE small batch)

Take 16 sequences. Train on only those, again and again. A healthy model **memorises them** and the loss goes close to 0.
If it cannot, the problem is in the model or the training loop, and more data will not fix it.

In [ ]:
# ==============================================================================
# Cell 6 : Overfit one batch (LSTM and transformer)
# ==============================================================================
def overfit_one_batch(model, steps=OVERFIT_STEPS, lr=1e-3, n_seqs=16):
    disable_dropout(model)
    model.train()
    batch = next(iter(make_loader(train_seqs[:n_seqs], tokenizer, batch_size=n_seqs, shuffle=False))).to(device)
    inp, tgt = shift(batch)
    n_real = (tgt != PAD_ID).sum().item()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    losses = []
    for _ in range(steps):
        opt.zero_grad()
        loss_sum = F.cross_entropy(logits_of(model, inp).reshape(-1, VOCAB_SIZE), tgt.reshape(-1),
                                   ignore_index=PAD_ID, reduction="sum")
        (loss_sum / n_real).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss_sum.item() / n_real)
    return losses


fig, ax = plt.subplots(figsize=(7, 4))
for name, builder, lr in (("lstm", build_lstm, 1e-3), ("transformer", build_transformer, 6e-4)):
    set_seed(SEED)
    m, _ = builder()
    losses = overfit_one_batch(m, lr=lr)
    ax.plot(losses, label=f"{name} (end {losses[-1]:.3f})")
    check(f"{name}: overfits one batch (final loss < 1.0 nats/token)", losses[-1] < 1.0,
          f"start {losses[0]:.2f} -> end {losses[-1]:.3f} after {OVERFIT_STEPS} steps. "
          "If it is still falling, raise OVERFIT_STEPS to 1000 before suspecting a bug")
ax.set_xlabel("step"); ax.set_ylabel("nats / token"); ax.set_title("Overfit one batch (should go near 0)")
ax.legend(); fig.tight_layout(); fig.savefig(OUT_DIR / "overfit_one_batch.png", dpi=150); plt.show()

## Part 3 — Does the model beat dumb baselines?

A score alone means nothing. 2.7 nats/token is good or bad **depending on the baselines**.
Expected order (lower is better): `uniform > unigram > bigram > LSTM >= transformer`.

Smoothing note: the unigram and bigram use **add-one (Laplace) smoothing** over the 21 things a model can predict (20 amino acids + EOS), so no probability is exactly 0 and the score is finite. This is a documented choice, not a hidden epsilon.

In [ ]:
# ==============================================================================
# Cell 7 : Baselines (uniform, unigram, bigram) and trained models, all on the SAME val set
# ==============================================================================
LUT = np.full(256, -1, dtype=np.int64)
for i, a in enumerate(STANDARD_AA):
    LUT[ord(a)] = i
TARGETS = list(range(20)) + [EOS_ID]          # what a model can predict: 20 AA + EOS


def pair_ids(seqs):
    """All (previous, next) token pairs as one flat index prev * 23 + next, with BOS and EOS added."""
    out = []
    for s in seqs:
        ids = LUT[np.frombuffer(s.encode("ascii"), dtype=np.uint8)]
        full = np.concatenate(([BOS_ID], ids, [EOS_ID]))
        out.append(full[:-1] * VOCAB_SIZE + full[1:])
    return np.concatenate(out)


rng_b = np.random.default_rng(SEED)
base_idx = rng_b.permutation(len(train_seqs))[:BASELINE_TRAIN_MAX]
base_train = [train_seqs[i] for i in base_idx]

counts = np.zeros(VOCAB_SIZE * VOCAB_SIZE, dtype=np.int64)
for start in range(0, len(base_train), 10_000):
    counts += np.bincount(pair_ids(base_train[start:start + 10_000]), minlength=VOCAB_SIZE * VOCAB_SIZE)
C = counts.reshape(VOCAB_SIZE, VOCAB_SIZE)                    # C[prev, next]

# unigram: how often each target token occurs (BOS is never a target)
uni_counts = C[:, TARGETS].sum(axis=0).astype(np.float64)
uni_p = np.zeros(VOCAB_SIZE)
uni_p[TARGETS] = (uni_counts + 1) / (uni_counts.sum() + len(TARGETS))

# bigram: P(next | prev) with add-one smoothing over the 21 targets
big_p = np.zeros((VOCAB_SIZE, VOCAB_SIZE))
for prev in [BOS_ID] + list(range(20)):
    row = C[prev, TARGETS].astype(np.float64)
    big_p[prev, TARGETS] = (row + 1) / (row.sum() + len(TARGETS))

val_pairs = pair_ids(val_seqs)
prev_v, next_v = val_pairs // VOCAB_SIZE, val_pairs % VOCAB_SIZE
n_tok = len(val_pairs)
uniform_nats = math.log(len(TARGETS))
unigram_nats = float(-np.log(uni_p[next_v]).sum() / n_tok)
bigram_nats = float(-np.log(big_p[prev_v, next_v]).sum() / n_tok)

# check: val token count matches what the model loss counts
tot = 0
for batch in make_loader(val_seqs, tokenizer, batch_size=256, shuffle=False):
    tot += (shift(batch)[1] != PAD_ID).sum().item()
check("baseline token count = model token count (same val tokens)", tot == n_tok, f"{tot:,} vs {n_tok:,}")

rows = [("uniform over 21 targets", uniform_nats), ("unigram (composition)", unigram_nats), ("bigram", bigram_nats)]

# trained models (optional: only if a checkpoint exists)
loaded = {}
for name, builder, fname in (("lstm", build_lstm, "lstm_best.pt"), ("transformer", build_transformer, "transformer_best.pt")):
    path = PATHS["models"] / fname
    if not path.exists():
        print(f"(no checkpoint {path.name}: {name} skipped)")
        continue
    try:
        m, _ = builder()
        m.load_state_dict(torch.load(path, map_location=device))
        m.eval()
        loaded[name] = m
        rows.append((name, mean_nats(m, val_seqs)[0]))
    except Exception as e:
        print(f"Could not load {path.name}: {e}")

table = pd.DataFrame(rows, columns=["model", "val nats/token"])
table["perplexity"] = np.exp(table["val nats/token"])
display(table.round(3))
nats = dict(rows)

check("order: uniform > unigram", nats["uniform over 21 targets"] > nats["unigram (composition)"],
      kind="should" if DATA_VERSION == "toy" else "must")
check("order: unigram > bigram", nats["unigram (composition)"] > nats["bigram"],
      "if this fails on TOY data it is normal: toy sequences are random", kind="should")
for name in loaded:
    check(f"{name} beats the bigram baseline", nats[name] < nats["bigram"],
          f"{nats[name]:.3f} vs {nats['bigram']:.3f}")
if {"lstm", "transformer"} <= set(loaded):
    check("transformer <= LSTM (expected, not guaranteed)", nats["transformer"] <= nats["lstm"] + 1e-3, kind="should")
if DATA_VERSION == "toy":
    print(f"\nTOY data is uniform random: the best possible score is about ln(20) = {math.log(20):.3f}. "
          "A model far BELOW that has a leak.")

## Part 4 — Are the generated sequences good, and NEW?

We compute the same numbers on **real validation sequences** (the reference row). A good generator is **close to the real row**, not better than it.

The most important column is **nearest train identity**: for each sequence, the identity (%) with its closest sequence in the train set.
- If many generated sequences are ≥ 90 % identical to a train sequence, the model **copies** its training data.
- The real-val row shows the same number for real sequences. It is high when the random split leaks near-duplicates (the limit explained in NB02).
- We only search a sample of the train set, so the numbers are a **lower bound** on copying.

In [ ]:
# ==============================================================================
# Cell 8 : Generate (if a checkpoint exists) and compute metrics for every set
# ==============================================================================
try:
    from rapidfuzz.process import cdist
    from rapidfuzz.distance import Levenshtein
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"], check=True)
    from rapidfuzz.process import cdist
    from rapidfuzz.distance import Levenshtein

GEN_CONFIGS = {
    "greedy": dict(greedy=True),
    "topk10_T0.8": dict(top_k=10, temperature=0.8),
    "nucleus_p0.95_T1.0": dict(top_p=0.95, temperature=1.0),
    "nucleus_p0.80_T1.2": dict(top_p=0.80, temperature=1.2),
}

rng_n = np.random.default_rng(SEED + 1)
nn_idx = rng_n.permutation(len(train_seqs))[:NN_TRAIN_SAMPLE]
nn_refs = [train_seqs[i] for i in nn_idx]
train_set = set(train_seqs)

p_data_dict = compute_aa_composition(base_train)
p_data = np.array([p_data_dict[a] for a in STANDARD_AA])
assert (p_data > 0).all()
HOMO = re.compile(r"(.)\1{4,}")                     # 5 or more identical letters in a row

# Part 5 helper: do the two conserved cysteines exist at roughly the right place?
# Windows are approximate (gaps are removed, so positions shift). Calibrate on the REAL row: it should be high there.
CYS1_WINDOW, CYS2_WINDOW = (15, 30), (85, 110)


def has_cys_pair(s):
    pos = [i + 1 for i, ch in enumerate(s) if ch == "C"]
    return any(CYS1_WINDOW[0] <= p <= CYS1_WINDOW[1] for p in pos) and any(CYS2_WINDOW[0] <= p <= CYS2_WINDOW[1] for p in pos)


def nearest_identity(queries, refs, chunk=50):
    """Best normalised Levenshtein similarity (= identity) of each query against all refs."""
    best = np.zeros(len(queries))
    for i in range(0, len(queries), chunk):
        d = cdist(queries[i:i + chunk], refs, scorer=Levenshtein.normalized_similarity, dtype=np.float32, workers=-1)
        best[i:i + chunk] = d.max(axis=1)
    return best


def report_row(name, seqs, reached=None):
    n = len(seqs)
    L = np.array([len(s) for s in seqs])
    comp = np.array([sum(s.count(a) for s in seqs) for a in STANDARD_AA], dtype=np.float64)
    pg = comp / comp.sum()
    kl = float(np.sum(np.where(pg > 0, pg * np.log(np.where(pg > 0, pg, 1.0) / p_data), 0.0)))
    ident = nearest_identity(seqs, nn_refs)
    return {
        "set": name, "n": n,
        "valid %": 100 * np.mean([is_valid_sequence(s) for s in seqs]),
        "reached EOS %": 100 * (np.mean(reached) if reached is not None else 1.0),
        "len mean": L.mean(), "len std": L.std(),
        "unique %": 100 * len(set(seqs)) / n,
        "KL comp (nats)": kl,
        "run>=5 %": 100 * np.mean([bool(HOMO.search(s)) for s in seqs]),
        "distinct AA / seq": np.mean([len(set(s)) for s in seqs]),
        "exact copy of train %": 100 * np.mean([s in train_set for s in seqs]),
        "nearest train id (median)": float(np.median(ident)),
        "identity >= 0.9 %": 100 * np.mean(ident >= 0.9),
        "cys pair %": 100 * np.mean([has_cys_pair(s) for s in seqs]),
    }


sets, reports = {}, []
real_ref = val_seqs[:N_GEN]
sets["real_val"] = real_ref
reports.append(report_row("real_val", real_ref))

if MODEL_TO_GENERATE in loaded:
    for cname, cfg in GEN_CONFIGS.items():
        print(f"generating {cname} ({N_GEN} sequences)...")
        r = generate(loaded[MODEL_TO_GENERATE], N_GEN, seed=SEED, **cfg)
        sets[cname] = r["seqs"]
        reports.append(report_row(cname, r["seqs"], r["reached_eos"]))
        with open(OUT_DIR / f"{cname}.fasta", "w", encoding="utf-8") as f:
            for i, s in enumerate(r["seqs"]):
                f.write(f">{cname}_{i}\n{s}\n")
else:
    print(f"No checkpoint for '{MODEL_TO_GENERATE}': only the real reference row is computed. Train the model first (NB03/NB04).")

rep = pd.DataFrame(reports).set_index("set")
display(rep.round(2).T)
rep.to_csv(OUT_DIR / "generation_report.csv")

In [ ]:
# ==============================================================================
# Cell 9 : Checks on the generated sets + length histogram
# ==============================================================================
real = rep.loc["real_val"]
for cname in [c for c in rep.index if c != "real_val"]:
    g = rep.loc[cname]
    check(f"{cname}: no PAD/BOS (valid = length in bounds and 20 letters)", g["valid %"] >= 99.0, f"{g['valid %']:.1f} % valid")
    check(f"{cname}: model learned to stop (reached EOS)", g["reached EOS %"] >= 95.0, f"{g['reached EOS %']:.1f} %", kind="should")
    check(f"{cname}: composition close to data (KL <= 0.05 nats)", g["KL comp (nats)"] <= 0.05, f"{g['KL comp (nats)']:.4f}", kind="should")
    check(f"{cname}: not copying train more than real val does (identity>=0.9, +10 pts margin)",
          g["identity >= 0.9 %"] <= real["identity >= 0.9 %"] + 10, f"{g['identity >= 0.9 %']:.1f} % vs real {real['identity >= 0.9 %']:.1f} %", kind="should")
    check(f"{cname}: no mass repetition (run>=5 close to real)", g["run>=5 %"] <= real["run>=5 %"] + 10,
          f"{g['run>=5 %']:.1f} % vs real {real['run>=5 %']:.1f} %", kind="should")

fig, ax = plt.subplots(figsize=(9, 4))
for name, seqs in sets.items():
    L = [len(s) for s in seqs]
    ax.hist(L, bins=np.arange(0, MAX_LEN + 5, 2), histtype="step", density=True, label=name, linewidth=2 if name == "real_val" else 1)
ax.set_xlabel("length (residues)"); ax.set_ylabel("density"); ax.set_title("Length distribution: generated vs real")
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT_DIR / "lengths_generated_vs_real.png", dpi=150); plt.show()

# 15 raw examples per set, to LOOK at them (qualitative check)
for name, seqs in sets.items():
    print(f"\n--- {name} (first 5) ---")
    for s in seqs[:5]:
        print(f"{len(s):3d}  {s}")

## Part 5 — Do they look like real VHH?

1. **Cysteine pair** (already in the table, column `cys pair %`): VHH have two conserved cysteines that make a disulfide bond. Check the **real** row first: it should be high. If it is low, the position windows in Cell 8 need to be calibrated on your real data (use the PWM from NB01).
2. **ANARCI** (optional): a tool that tries to number an antibody sequence. A real VHH is numbered, random text is not. Compare the % numbered for generated vs real.
3. **Structure tools** (NanoBodyBuilder2 / ESMFold): only on a **sample of about 100** sequences, and only after steps 1 and 2 look fine.

In [ ]:
# ==============================================================================
# Cell 10 : ANARCI (optional) - % of sequences that can be numbered (IMGT)
# ==============================================================================
try:
    from anarci import anarci
    ANARCI_OK = True
except Exception:
    ANARCI_OK = False
    print("ANARCI is not installed: skipped (optional). On Colab it needs HMMER (conda); ask the team before spending time on it.")

if ANARCI_OK:
    arows = []
    for name, seqs in sets.items():
        sample = [s for s in seqs[:ANARCI_N] if s]
        try:
            numbering, _, _ = anarci([(f"s{i}", s) for i, s in enumerate(sample)], scheme="imgt", output=False)
            arows.append({"set": name, "n": len(sample), "numbered %": 100 * np.mean([x is not None for x in numbering])})
        except Exception as e:
            print(f"ANARCI failed on {name}: {e}")
    if arows:
        adf = pd.DataFrame(arows).set_index("set")
        display(adf.round(1))
        real_n = adf.loc["real_val", "numbered %"] if "real_val" in adf.index else None
        for name in adf.index:
            if name != "real_val" and real_n is not None:
                check(f"{name}: ANARCI numbering rate close to real", adf.loc[name, "numbered %"] >= real_n - 15,
                      f"{adf.loc[name, 'numbered %']:.0f} % vs real {real_n:.0f} %", kind="should")

In [ ]:
# ==============================================================================
# Cell 11 : Final summary
# ==============================================================================
summary = pd.DataFrame(CHECKS)
summary.to_csv(OUT_DIR / "checks_summary.csv", index=False)
pd.set_option("display.max_colwidth", 120)
display(summary)

status_counts = summary["status"].value_counts().to_dict()
print(f"\nPASS: {status_counts.get('PASS', 0)} | WARN: {status_counts.get('WARN', 0)} | FAIL: {status_counts.get('FAIL', 0)}")
if status_counts.get("FAIL", 0):
    print("FAIL means a bug. Fix it before you train or generate anything.")
elif status_counts.get("WARN", 0):
    print("No bug found, but WARN lines are suspicious: read each one and explain it in the report.")
else:
    print("All checks passed.")
print(f"Saved in: {OUT_DIR}")

## How to read the results (for the report)

| If you see | It means | What to do |
|---|---|---|
| Part 1 FAIL | a code bug (mask, loss, sampling) | fix, rerun Part 1 |
| Cannot overfit one batch | model or training loop is broken | check the loss, the shift, the learning rate, the init |
| Model not better than bigram | it learned almost nothing | train longer, check data, lower the learning rate |
| `exact copy of train %` or `identity >= 0.9 %` very high | memorisation | fewer epochs, more dropout, more data, cluster split |
| `reached EOS %` low | the model never learned to stop | check that EOS is in the targets and not masked as PAD |
| `run>=5 %` high with greedy | degeneration (repeats) | expected for greedy (Holtzman et al. 2020): confirm with the numbers |
| `cys pair %` far below real | the model misses the VHH backbone | train longer, look at the conserved positions |

**Limits to write in the report:** random split (near-duplicates leak, so val scores are optimistic); the nearest-copy search uses only a sample of train; the cysteine windows are approximate (no real alignment yet); no structure check yet.